# NB 1.1 &mdash; Presa de contacte amb les dades

**MP 5134 &mdash; Disseny i avaluació de models basats en aprenentatge automàtic**
UT1: Entorn de treball i primer model de principi a fi

*Versió amb el conjunt **California Housing**, el mateix que fa servir Aurélien
Géron al capítol 2 de* Hands-On Machine Learning*.*

---

### Què farem avui

Res d'entrenar models. Avui només mirem dades.

Sembla poc, però és la meitat de la feina real d'un projecte d'aprenentatge
automàtic. Si no entens què hi ha dins del fitxer, qualsevol model que entrenis
serà una loteria. I com veuràs al final d'aquest notebook, aquest conjunt en
concret amaga un parell de sorpreses que només es veuen mirant.

En acabar has de saber respondre:

1. Quantes mostres i quantes característiques tenim?
2. Què volem predir exactament?
3. Quines columnes tenen valors absents, i quantes categories té la columna de text?
4. Què li passa a la variable objectiu que hauria de fer-nos desconfiar?

## 1. De què va tot això

L'aprenentatge automàtic no és més que això: **tenim exemples del passat i volem
fer prediccions sobre casos nous**.

La diferència amb la programació que ja coneixeu és on posem les regles.

| Programació clàssica | Aprenentatge automàtic |
|---|---|
| Tu escrius les regles | L'algorisme les dedueix dels exemples |
| Dades + regles &rarr; resposta | Dades + respostes &rarr; regles |

Si volguéssim estimar el preu d'un habitatge amb programació clàssica, hauríem
d'escriure nosaltres les condicions: *si té més de cinc habitacions i està a
menys d'un quilòmetre del mar, llavors...*. Amb aprenentatge automàtic li donem
milers d'exemples i deixem que l'algorisme trobi el patró.

Això té una conseqüència important: **el model només serà tan bo com les dades
que li donem**. D'aquí que avui dediquem tota una sessió a mirar-les.

## 2. El nostre conjunt de dades

**California Housing**, construït a partir del cens dels Estats Units de 1990. És
el conjunt amb què Géron obre el seu llibre, i s'ha convertit en un clàssic per
una raó: és prou senzill per entendre'l d'una ullada i prou brut per ensenyar
tots els problemes reals.

Aquí hi ha una cosa que convé tenir clara des del principi. **Cada fila no és un
habitatge**, sinó un *districte censal*: una zona amb entre 600 i 3.000
habitants. Els valors són agregats de tot el districte.

| Columna | Què és |
|---|---|
| `longitude`, `latitude` | coordenades del districte |
| `housing_median_age` | antiguitat mediana dels habitatges |
| `total_rooms` | nombre total d'habitacions **del districte** |
| `total_bedrooms` | nombre total de dormitoris del districte |
| `population` | habitants del districte |
| `households` | nombre de llars |
| `median_income` | ingressos mitjans (en desenes de milers de dòlars) |
| `median_house_value` | **preu mitjà dels habitatges (dòlars)** |
| `ocean_proximity` | proximitat al mar (text) |

Que siguin totals de districte i no valors per habitatge té conseqüències: un
districte gran tindrà `total_rooms` alt simplement perquè hi viu més gent, no
perquè les cases siguin més grans. Hi tornarem.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

URL_DADES = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
              "master/datasets/housing/housing.csv")

df = pd.read_csv(URL_DADES)
print("Dades carregades correctament.")

`pd.read_csv` accepta directament una URL: no cal descarregar res a mà, i això
fa que el notebook funcioni igual a qualsevol ordinador amb connexió.

### 2.1 Quantes dades tenim

In [ ]:
files, columnes = df.shape
print(f"Files:    {files}")
print(f"Columnes: {columnes}")

Aquí apareix el **vocabulari fonamental** del mòdul. Val la pena fixar-lo ara,
perquè el farem servir cada dia fins al maig:

- **Mostra** o **instància**: una fila. Aquí, un districte censal.
- **Característica** (*feature*): una columna que fem servir com a entrada.
- **Variable objectiu** (*target*): la columna que volem predir.

Una forma útil de recordar-ho: les mostres són *els exemples*, les
característiques són *el que sabem de cada exemple*, i la variable objectiu és
*el que volem endevinar*.

Compte amb una confusió molt freqüent: **no totes les columnes són
característiques**. `median_house_value` és la variable objectiu i no entrarà mai com a
entrada del model.

In [ ]:
df.head()

`head()` mostra les cinc primeres files. És el gest més repetit de tot el curs:
abans de fer res amb unes dades, mira-les.

In [ ]:
df.info()

`info()` dóna tres coses de cop: el **tipus** de cada columna, quants valors **no
nuls** té cadascuna, i la memòria que ocupa.

Dues coses per fixar-s'hi:

**`ocean_proximity` és de tipus `object`**, és a dir, text. Els algorismes de
scikit-learn només accepten números, així que aquesta columna no la podem fer
servir tal com està. Convertir-la serà feina de la **UT3**.

**`total_bedrooms` té menys valors no nuls que la resta.** Això vol dir buits.

In [ ]:
df.describe()

### 2.2 Llegir un `describe()`

Aquesta taula sembla àrida però diu moltíssim. Mira-la amb aquestes preguntes:

**Els mínims i màxims tenen sentit?** Fixa't en `median_house_value`: el màxim és
exactament **500.001**. Un número així de rodó, i acabat en 1, no apareix per
casualitat en dades reals.

**La mitjana i la mediana (50%) s'assemblen?** Si són molt diferents, la
distribució està esbiaixada. Mira `total_rooms`: la mitjana és molt superior a la
mediana perquè hi ha districtes enormes que estiren la mitjana cap amunt.

**Els quartils estan on t'esperes?** El 25% i el 75% et diuen com es reparteixen
els valors sense necessitat de dibuixar res.

## 3. Valors absents i categories

Cap conjunt de dades real està complet.

In [ ]:
absents = df.isna().sum()
print(absents[absents > 0])
print()
print(f"Files completes: {df.dropna().shape[0]} de {df.shape[0]}")

Només una columna té buits, i afecten poc més de l'1% de les files.

Això és un problema pràctic immediat: **la majoria d'algorismes de scikit-learn
no accepten valors absents**. Si li passes una taula amb buits, peta.

Hi ha tres sortides possibles, i cadascuna té un cost:

1. **Eliminar les files** amb buits. Simple, però perds mostres.
2. **Eliminar la columna** sencera. Perds informació potencialment útil.
3. **Imputar**: omplir els buits amb la mitjana, la mediana o un valor estimat.

Avui farem servir l'opció 1 perquè és la més directa. A la **UT3** hi tornarem
amb calma, perquè la decisió no és innocent: imputar malament pot fer que el
model aprengui coses que no són certes.

In [ ]:
df["ocean_proximity"].value_counts()

### 3.1 Una categoria que gairebé no existeix

Cinc categories, però fixa't en `ISLAND`: té un grapat de mostres comptades.

Això és una **categoria minoritària**, i dóna problemes reals. Si en fer la
partició entre entrenament i prova totes les mostres d'`ISLAND` cauen al mateix
costat, el model o bé no l'haurà vista mai, o bé no la podrà avaluar.

És un cas concret d'un problema més general que treballarem a la **UT10**: com
partir les dades sense que les categories rares es perdin pel camí.

## 4. Què volem predir

La variable objectiu és **`median_house_value`**, el preu mitjà dels habitatges
del districte. És un número continu, així que tenim un problema de **regressió**.

Però abans de continuar, tornem a aquell 500.001 sospitós.

In [ ]:
topall = df["median_house_value"].max()
al_topall = (df["median_house_value"] >= topall).sum()

print(f"Valor màxim: {topall:,.0f} $")
print(f"Districtes amb aquest valor exacte: {al_topall}")
print(f"Són el {100 * al_topall / len(df):.1f} % del total")

### 4.1 La primera sorpresa

Gairebé un 5% dels districtes tenen exactament el mateix preu, i és el màxim.

Això no és una coincidència: quan es van recollir les dades, **es va decidir
truncar els valors** per damunt de mig milió. Un districte que en realitat valia
800.000 apareix com a 500.001.

La conseqüència és directa i important: **el model no podrà predir mai res per
damunt d'aquest topall**, perquè als exemples que li donem no n'hi ha cap.
Aprendrà que existeix un sostre que en realitat no existeix.

Què fem amb això? En un projecte real hi hauria dues opcions: aconseguir les
dades correctes d'aquells districtes, o bé eliminar-los i acceptar que el model
només val per a preus per sota de mig milió. Cap de les dues és gratuïta.

Ho deixem així de moment, però recorda-ho. És el tipus de detall que no apareix a
cap mètrica i que només es veu mirant les dades.

### 4.2 Fabricar un problema de classificació

El conjunt només porta variable objectiu numèrica, però podem construir-ne una de
categòrica per treballar també la classificació: **és un districte car?**

Compte, que aquest llindar és una **decisió nostra**, no ve amb les dades. Amb un
altre valor tindríem un problema diferent.

In [ ]:
LLINDAR = 350_000
df["expensive"] = (df["median_house_value"] >= LLINDAR).astype(int)

proporcio = df["expensive"].value_counts(normalize=True).sort_index()
print(proporcio)
print()
print(f"Districtes cars: {100 * proporcio[1]:.1f} %")

### 4.3 La segona sorpresa, i serà clau a la UT4

Els districtes cars són una minoria clara.

Això es diu **desbalanç de classes** i té una conseqüència que sorprèn molt: un
model que digui sempre *"no és car"*, sense mirar cap dada, encertaria la gran
majoria de vegades.

Guarda aquest número. Al NB 1.2 el veuràs en acció, i a la **UT4** el farem servir
per demostrar per què el percentatge d'encerts és una mètrica que enganya.

## 5. Primeres gràfiques

Les taules diuen molt, però els ulls detecten coses que cap `describe()` et
donarà.

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df["median_house_value"], bins=60, edgecolor="white")
plt.xlabel("Preu mitjà de l'habitatge ($)")
plt.ylabel("Nombre de districtes")
plt.title("Distribució de la variable objectiu")
plt.show()

Un **histograma** reparteix els valors en intervals i compta quants n'hi ha a
cadascun.

Aquí el truncament es veu a simple vista: hi ha una **barra aïllada a l'extrem
dret**, molt més alta del que li tocaria. Això és l'acumulació artificial de tots
els districtes que valien més de mig milió.

Aquesta és la millor demostració de per què val la pena dibuixar les dades. El
`describe()` ja donava la pista amb aquell màxim rodó, però la gràfica ho fa
evident en un segon.

In [ ]:
plt.figure(figsize=(8, 7))
dispersio = plt.scatter(df["longitude"], df["latitude"],
                        c=df["median_house_value"], s=4,
                        alpha=0.35, cmap="viridis")
plt.colorbar(dispersio, label="Preu mitjà ($)")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("On són els districtes cars?")
plt.show()

Dibuixant longitud contra latitud apareix **la silueta de Califòrnia**, sense
haver carregat cap mapa: només són dues columnes de números.

I el color revela un patró claríssim: **la costa és cara i l'interior és barat**.

Això té una lectura pràctica. La latitud i la longitud, per si soles, no
signifiquen res per a un model. Però combinades porten moltíssima informació
sobre el preu. És un primer avís d'una idea que treballarem a la **UT3**: sovint
el que importa no són les variables per separat sinó la relació entre elles.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(df["median_income"], df["median_house_value"], s=3, alpha=0.15)
plt.xlabel("Ingressos mitjans (desenes de milers $)")
plt.ylabel("Preu mitjà de l'habitatge ($)")
plt.title("Hi ha relació entre ingressos i preu?")
plt.show()

El núvol s'estira en diagonal: als districtes amb ingressos alts, els habitatges
són cars. Serà segurament la característica més útil del conjunt.

Però mira bé i veuràs dues coses més:

**La línia horitzontal de dalt** és el truncament dels 500.001 que ja coneixem.

**Hi ha altres línies horitzontals més febles**, cap als 450.000 i els 350.000.
Són artefactes del procés de recollida de dades. Géron els comenta al llibre i
recomana plantejar-se si convé eliminar aquests districtes perquè el model no
n'aprengui el patró.

Aquesta idea de mesurar quant es relacionen dues variables té nom propi i és tot
el contingut de la **UT3**: la correlació.

## 6. Exercicis

**1.** Quants districtes hi ha de cada categoria d'`ocean_proximity`? Calcula el
preu mitjà de cadascuna amb
`df.groupby("ocean_proximity")["median_house_value"].mean()`. Quadra amb el que
has vist al mapa?

**2.** Crea una columna nova `rooms_per_household` dividint `total_rooms` entre
`households`. Per què aquesta variable té més sentit que `total_rooms` a soles?

**3.** Dibuixa l'histograma de `housing_median_age`. Hi trobes el mateix problema
que a la variable objectiu? Quin és el valor màxim i quantes mostres hi ha?

**4.** Quants districtes tenen més de 10 habitacions per llar? Mira'ls amb
`df[df["rooms_per_household"] > 10]`. Et semblen creïbles?

**5.** Pensa i escriu la resposta: si només poguessis fer servir **tres** columnes
per predir el preu, quines triaries i per què? Ho comprovarem a la UT3.